# minRNN — Foundations (minGRU / minLSTM)

Self-contained notebook for *Were RNNs All We Needed?* (Feng et al., 2024).

minGRU and minLSTM as the linear recurrence `h_t = a_t * h_{t-1} + b_t`,
computed three ways (sequential, parallel log-space, vanilla), plus a correctness harness.

GPU on Colab: **Runtime > Change runtime type > GPU**.


## 1. Environment

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device =", dev)
if dev == "cuda":
    print("gpu  =", torch.cuda.get_device_name(0))

torch 2.11.0+cu130 | device = cuda
gpu  = Tesla T4


## 2. Scan primitives

The shared recurrence engine every model reuses.

In [ ]:
# sequential recurrence for h
def naive_sequential_scan(a, b, h0=None):
    B, T, D = a.shape # batch, time, dimension
    h_prev = torch.zeros(B, D, dtype=a.dtype, device=a.device) if h0 is None else h0
    outs = []
    for t in range(T):
        h_prev = a[:, t] * h_prev + b[:, t]
        outs.append(h_prev)
    return torch.stack(outs, dim=1)

# parallel recurrence for h, but vanishes
def vanilla_scan(a, b, h0=None):
    A = torch.cumprod(a, dim=1)
    inner = torch.cumsum(b / A, dim=1)
    if h0 is not None:
        inner = inner + h0.unsqueeze(1)
    return A * inner

# parallel recurrence for h in log space, solving vanish
def heinsen_associative_scan_log(log_coeffs, log_values):
    a_star = log_coeffs.cumsum(dim=1)
    log_h0_plus_b_star = torch.logcumsumexp(log_values - a_star, dim=1)
    return (a_star + log_h0_plus_b_star).exp()

# g is the positive activation that replaces tanh; log_g is its log
# enforces constraints later on log(x), x>= 0
def g(x):
    return torch.where(x >= 0, x + 0.5, torch.sigmoid(x))

def log_g(x):
    return torch.where(x >= 0, (F.relu(x) + 0.5).log(), -F.softplus(-x))



## 3. minGRU

Gates depend on the current input, not the previous hidden state, which is what makes it parallelizable.

In [ ]:
# dealing with base case h0
def _scan_log_with_h0(log_a, log_b, h0=None):
    B, T, D = log_a.shape
    log_a = F.pad(log_a, (0, 0, 1, 0))
    if h0 is None:
        log_h0 = torch.full((B, 1, D), float("-inf"), dtype=log_b.dtype, device=log_b.device)
    else:
        log_h0 = h0.clamp(min=1e-20).log().unsqueeze(1)
    log_b = torch.cat([log_h0, log_b], dim=1)
    return heinsen_associative_scan_log(log_a, log_b)[:, 1:]


class minGRU(nn.Module):
    def __init__(self, dim, hidden_dim=None, proj_out=False):
        super().__init__()
        hidden_dim = hidden_dim or dim
        self.dim, self.hidden_dim = dim, hidden_dim
        self.to_z = nn.Linear(dim, hidden_dim, bias=False)
        self.to_h = nn.Linear(dim, hidden_dim, bias=False)
        self.to_out = nn.Linear(hidden_dim, dim, bias=False) if proj_out else nn.Identity()

    def _pre(self, x):
        return self.to_z(x), self.to_h(x)

    def forward_parallel_log(self, x, h0=None):
        pre_z, pre_h = self._pre(x)
        log_a = -F.softplus(pre_z)
        log_z = -F.softplus(-pre_z)
        log_b = log_z + log_g(pre_h)
        return self.to_out(_scan_log_with_h0(log_a, log_b, h0))

    def forward_parallel_vanilla(self, x, h0=None):
        pre_z, pre_h = self._pre(x)
        z = pre_z.sigmoid()
        return self.to_out(vanilla_scan(1.0 - z, z * g(pre_h), h0))

    def forward_sequential(self, x, h0=None):
        pre_z, pre_h = self._pre(x)
        z = pre_z.sigmoid()
        return self.to_out(naive_sequential_scan(1.0 - z, z * g(pre_h), h0))

    def forward(self, x, h0=None, mode="parallel_log"):
        return {"parallel_log": self.forward_parallel_log,
                "parallel_vanilla": self.forward_parallel_vanilla,
                "sequential": self.forward_sequential}[mode](x, h0)


## 4. minLSTM

Same structure; forget/input gates are normalized to sum to 1.

In [ ]:
class minLSTM(nn.Module):
    def __init__(self, dim, hidden_dim=None, proj_out=False):
        super().__init__()
        hidden_dim = hidden_dim or dim
        self.dim, self.hidden_dim = dim, hidden_dim
        self.to_f = nn.Linear(dim, hidden_dim, bias=False)
        self.to_i = nn.Linear(dim, hidden_dim, bias=False)
        self.to_h = nn.Linear(dim, hidden_dim, bias=False)
        self.to_out = nn.Linear(hidden_dim, dim, bias=False) if proj_out else nn.Identity()

    def _pre(self, x):
        return self.to_f(x), self.to_i(x), self.to_h(x)

    def forward_parallel_log(self, x, h0=None):
        pre_f, pre_i, pre_h = self._pre(x)
        log_f = -F.softplus(-pre_f)
        log_i = -F.softplus(-pre_i)
        log_denom = torch.logaddexp(log_f, log_i)
        log_a = log_f - log_denom
        log_b = (log_i - log_denom) + log_g(pre_h)
        return self.to_out(_scan_log_with_h0(log_a, log_b, h0))

    def _ab(self, x):
        pre_f, pre_i, pre_h = self._pre(x)
        f, i = pre_f.sigmoid(), pre_i.sigmoid()
        denom = f + i
        return f / denom, (i / denom) * g(pre_h)

    def forward_parallel_vanilla(self, x, h0=None):
        a, b = self._ab(x)
        return self.to_out(vanilla_scan(a, b, h0))

    def forward_sequential(self, x, h0=None):
        a, b = self._ab(x)
        return self.to_out(naive_sequential_scan(a, b, h0))

    def forward(self, x, h0=None, mode="parallel_log"):
        return {"parallel_log": self.forward_parallel_log,
                "parallel_vanilla": self.forward_parallel_vanilla,
                "sequential": self.forward_sequential}[mode](x, h0)


## 5. Selective Copying Task

In [ ]:
# Generates B shuffled batches of length seq_len with data_len tokens of interest from (0, vocab), and remaining noise tokens 0
def generate_batch(B, seq_len = 4096, data_len = 16, vocab = 16, device = dev):
    # generate random indices for data tokens, shuffle into noise tokens
    pos = torch.rand(B, seq_len, device = device).argsort(dim = 1)[:, :data_len].sort(dim = 1).values
    data = torch.randint(1, vocab, (B, data_len), device = device)
    x = torch.zeros(B, seq_len + data_len, dtype = torch.long, device = device)
    x.scatter_(1, pos, data)

    x[:, seq_len:] = vocab
    return x, data


In [ ]:
class Model(nn.Module):
    def __init__(self, cell, n_layers, vocab = 16, dim = 64, expansion = 6, dropout = 0.1):
        super().__init__()

        # convert discrete tokens to vector i.e. factor() in R
        self.emb = nn.Embedding(vocab + 1, dim)
        # layer normalization - some type of normalization is used but unspecified
        self.ln = nn.ModuleList([nn.LayerNorm(dim) for _ in range(n_layers)])
        # state expansion factor of 6 specified
        self.lyrs = nn.ModuleList([cell(dim, hidden_dim = dim * expansion, proj_out = True) for _ in range(n_layers)])
        # dropout of 0.1 specified
        self.drop = nn.Dropout(dropout)
        self.output = nn.Linear(dim, vocab)

    def forward(self, x):
        h = self.emb(x)
        for norm, layer in zip(self.ln, self.lyrs):
            # residual connection specified
            h = h + self.drop(layer(norm(h)))
        return self.output(h)


def train(cell, n_layers, steps, seq_len = 4096, batch = 64, accum = 2, lr = 3e-4, clip = 1.0):
    torch.manual_seed(0)
    model = Model(cell, n_layers).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr = lr)
    t0 = time.time()

    for step in range(1, steps + 1):
        opt.zero_grad(set_to_none = True)
        # gradient accumulation, split into smaller sub-batches for memory
        for a in range(accum):
            x, y = generate_batch(batch // accum, seq_len = seq_len, device = dev)
            # predict and keep only outputs
            y_hat = model(x)[:, -y.size(1):]

            loss = F.cross_entropy(y_hat.reshape(-1, 16), y.reshape(-1))
            (loss / accum).backward()
        # gradient clipping specified
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()

        if step % 100 == 0:
            print(f"Step {step}/{steps} | Loss: {loss.item():.4f} | {(time.time() - t0) / step:.2f}s/step")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for _ in range(8):
            x, y = generate_batch(32, seq_len = seq_len, device = dev)
            y_hat = model(x)[:, -y.size(1):].argmax(dim = -1)
            correct += (y_hat == y).sum().item()
            total += y.numel()
    return correct / total

In [ ]:
accuracy = train(minLSTM, 1, steps = 400000)
print(f"minLSTM (1 layer), Accuracy: {accuracy * 100:.2f}%")

accuracy = train(minLSTM, 2, steps = 400000)
print(f"minLSTM (2 layers), Accuracy: {accuracy * 100:.2f}%")

accuracy = train(minLSTM, 3, steps = 400000)
print(f"minLSTM (3 layers), Accuracy: {accuracy * 100:.2f}%")

accuracy = train(minGRU, 1, steps = 400000)
print(f"minGRU (1 layer), Accuracy: {accuracy * 100:.2f}%")

accuracy = train(minGRU, 2, steps = 400000)
print(f"minGRU (1 layer), Accuracy: {accuracy * 100:.2f}%")

accuracy = train(minGRU, 3, steps = 400000)
print(f"minGRU (1 layer), Accuracy: {accuracy * 100:.2f}%")

KeyboardInterrupt: 